# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 17 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

The embedded data reproduce the supplied Kaggle dataset using [this public CSV mirror](https://gist.github.com/keerthibotlagunta/c191da81998b3d728507108cde56fbab).

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
from io import StringIO

import numpy as np
import pandas as pd
import plotly.express as px

# The embedded copy makes the submission reproducible without a separate CSV.
csv_text = """Rank,CCA3,Country/Territory,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area (km²),Density (per km²),Growth Rate,World Population Percentage
36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,1.0257,0.52
138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.9957,0.04
34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,1.0164,0.56
213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.9831,0.0
203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,1.01,0.0
42,AGO,Angola,Luanda,Africa,35588987,33428485,28127721,23364185,16394062,11828638,8330047,6029700,1246700,28.5466,1.0315,0.45
224,AIA,Anguilla,The Valley,North America,15857,15585,14525,13172,11047,8316,6560,6283,91,174.2527,1.0066,0.0
201,ATG,Antigua and Barbuda,Saint John’s,North America,93763,92664,89941,85695,75055,63328,64888,64516,442,212.1335,1.0058,0.0
33,ARG,Argentina,Buenos Aires,South America,45510318,45036032,43257065,41100123,37070774,32637657,28024803,23842803,2780400,16.3683,1.0052,0.57
140,ARM,Armenia,Yerevan,Asia,2780469,2805608,2878595,2946293,3168523,3556539,3135123,2534377,29743,93.4831,0.9962,0.03
198,ABW,Aruba,Oranjestad,North America,106445,106585,104257,100341,89101,65712,62267,59106,180,591.3611,0.9991,0.0
55,AUS,Australia,Canberra,Oceania,26177413,25670051,23820236,22019168,19017963,17048003,14706322,12595034,7692024,3.4032,1.0099,0.33
99,AUT,Austria,Vienna,Europe,8939617,8907777,8642421,8362829,8010428,7678729,7547561,7465301,83871,106.5877,1.002,0.11
91,AZE,Azerbaijan,Baku,Asia,10358074,10284951,9863480,9237202,8190337,7427836,6383060,5425317,86600,119.6082,1.0044,0.13
176,BHS,Bahamas,Nassau,North America,409984,406471,392697,373272,325014,270679,223752,179129,13943,29.4043,1.0051,0.01
154,BHR,Bahrain,Manama,Asia,1472233,1477469,1362142,1213645,711442,517418,362595,222555,765,1924.4876,1.0061,0.02
8,BGD,Bangladesh,Dhaka,Asia,171186372,167420951,157830000,148391139,129193327,107147651,83929765,67541860,147570,1160.035,1.0108,2.15
186,BRB,Barbados,Bridgetown,North America,281635,280693,278083,274711,264657,258868,253575,241397,430,654.9651,1.0015,0.0
96,BLR,Belarus,Minsk,Europe,9534954,9633740,9700609,9731427,10256483,10428525,9817257,9170786,207600,45.9295,0.9955,0.12
81,BEL,Belgium,Brussels,Europe,11655930,11561717,11248303,10877947,10264343,9959560,9828986,9629376,30528,381.8111,1.0038,0.15
177,BLZ,Belize,Belmopan,North America,405272,394921,359871,322106,240406,182589,145133,120905,22966,17.6466,1.0131,0.01
77,BEN,Benin,Porto-Novo,Africa,13352864,12643123,10932783,9445710,6998023,5133419,3833939,3023443,112622,118.5635,1.0274,0.17
206,BMU,Bermuda,Hamilton,North America,64184,64031,63144,63447,61371,57470,53565,52019,54,1188.5926,1.0,0.0
165,BTN,Bhutan,Thimphu,Asia,782455,772506,743274,705516,587207,558442,415257,298894,38394,20.3796,1.0064,0.01
80,BOL,Bolivia,Sucre,South America,12224110,11936162,11090085,10223270,8592656,7096194,5736088,4585693,1098581,11.1272,1.012,0.15
137,BIH,Bosnia and Herzegovina,Sarajevo,Europe,3233526,3318407,3524324,3811088,4179350,4494310,4199820,3815561,51209,63.1437,0.9886,0.04
144,BWA,Botswana,Gaborone,Africa,2630296,2546402,2305171,2091664,1726985,1341474,938578,592244,582000,4.5194,1.0162,0.03
7,BRA,Brazil,Brasilia,South America,215313498,213196304,205188205,196353492,175873720,150706446,122288383,96369875,8515767,25.2841,1.0046,2.7
221,VGB,British Virgin Islands,Road Town,North America,31305,30910,29366,27556,20104,15617,11109,9581,151,207.3179,1.0059,0.0
175,BRN,Brunei,Bandar Seri Begawan,Asia,449002,441725,421437,396053,333926,261928,187921,133343,5765,77.8841,1.0081,0.01
108,BGR,Bulgaria,Sofia,Europe,6781953,6979175,7309253,7592273,8097691,8767778,8980606,8582950,110879,61.1654,0.9849,0.09
58,BFA,Burkina,Faso Ouagadougou,Africa,22673762,21522626,18718019,16116845,11882888,9131361,6932967,5611666,272967,83.0641,1.0259,0.28
78,BDI,Burundi,Bujumbura,Africa,12889576,12220227,10727148,9126605,6307659,5483793,4312834,3497834,27834,463.0874,1.027,0.16
73,KHM,Cambodia,Phnom Penh,Asia,16767842,16396860,15417523,14363532,12118841,8910808,6198959,6708525,181035,92.6221,1.0108,0.21
53,CMR,Cameroon,Yaounde,Africa,27914536,26491087,23012646,19878036,15091594,11430520,8519891,6452787,475442,58.7128,1.0263,0.35
39,CAN,Canada,Ottawa,North America,38454327,37888705,35732126,33963412,30683313,27657204,24511510,21434577,9984670,3.8513,1.0078,0.48
171,CPV,Cape Verde,Praia,Africa,593149,582640,552166,521212,458251,364563,317234,287262,4033,147.0739,1.0089,0.01
205,CYM,Cayman Islands,George Town,North America,68706,67311,60911,54074,39658,26027,17100,10533,264,260.25,1.0084,0.0
117,CAF,Central African Republic,Bangui,Africa,5579144,5343020,4819333,4660067,3759170,2809221,2415276,2067356,622984,8.9555,1.0224,0.07
69,TCD,Chad,N'Djamena,Africa,17723315,16644701,14140274,11894727,8259137,5827069,4408230,3667394,1284000,13.8032,1.0316,0.22
65,CHL,Chile,Santiago,South America,19603733,19300315,17870124,17004162,15351799,13342868,11469828,9820481,756102,25.9274,1.0057,0.25
1,CHN,China,Beijing,Asia,1425887337,1424929781,1393715448,1348191368,1264099069,1153704252,982372466,822534450,9706961,146.8933,1.0,17.88
28,COL,Colombia,Bogota,South America,51874024,50930662,47119728,44816108,39215135,32601393,26176195,20905254,1141748,45.4339,1.0069,0.65
163,COM,Comoros,Moroni,Africa,836774,806166,730216,656024,536758,431119,328328,242351,1862,449.3953,1.0184,0.01
223,COK,Cook Islands,Avarua,Oceania,17011,17029,17695,17212,15897,17123,17651,20470,236,72.0805,1.0005,0.0
124,CRI,Costa Rica,San José,North America,5180829,5123105,4895242,4622252,3979193,3158253,2414303,1855697,51100,101.3861,1.0052,0.06
130,HRV,Croatia,Zagreb,Europe,4030358,4096868,4254815,4368682,4548434,4873707,4680144,4492638,56594,71.2153,0.9927,0.05
85,CUB,Cuba,Havana,North America,11212191,11300698,11339894,11290417,11105791,10626680,9809107,8869636,109884,102.0366,0.9961,0.14
189,CUW,Curacao,Willemstad,North America,191163,189288,169572,159380,141424,155446,156851,150385,444,430.5473,1.0043,0.0
158,CYP,Cyprus,Nicosia,Europe,1251488,1237537,1187280,1129686,948237,788500,679327,640804,9251,135.2814,1.0059,0.02
88,CZE,Czech Republic,Prague,Europe,10493986,10530953,10523798,10464749,10234710,10301192,10270060,9795744,78865,133.0627,0.9984,0.13
115,DNK,Denmark,Copenhagen,Europe,5882261,5825641,5677796,5550849,5340655,5144623,5125392,4922963,43094,136.4984,1.0048,0.07
160,DJI,Djibouti,Djibouti,Africa,1120849,1090156,1006259,919199,742033,577173,324121,144379,23200,48.3125,1.0138,0.01
204,DMA,Dominica,Roseau,North America,72737,71995,70007,68755,68346,69481,72978,68895,751,96.8535,1.0045,0.0
84,DOM,Dominican Republic,Santo Domingo,North America,11228821,10999664,10405832,9775755,8540791,7129004,5755800,4475871,48671,230.7087,1.01,0.14
15,COD,DR Congo,Kinshasa,Africa,99010212,92853164,78656904,66391257,48616317,35987541,26708686,20151733,2344858,42.2244,1.0325,1.24
67,ECU,Ecuador,Quito,South America,18001000,17588595,16195902,14989585,12626507,10449837,8135845,6172215,276841,65.0229,1.0114,0.23
14,EGY,Egypt,Cairo,Africa,110990103,107465134,97723799,87252413,71371371,57214630,43748556,34781986,1002450,110.7188,1.0158,1.39
112,SLV,El Salvador,San Salvador,North America,6336392,6292731,6231066,6114034,5958482,5367179,4508992,3619090,21041,301.145,1.0035,0.08
152,GNQ,Equatorial Guinea,Malabo,Africa,1674908,1596049,1346973,1094524,684977,465549,282509,316955,28051,59.7094,1.0247,0.02
132,ERI,Eritrea,Asmara,Africa,3684032,3555868,3340006,3147727,2392880,2149960,1657982,1272748,117600,31.3268,1.0176,0.05
156,EST,Estonia,Tallinn,Europe,1326062,1329444,1314657,1331535,1396877,1570674,1476983,1361999,45227,29.3201,0.998,0.02
159,SWZ,Eswatini,Mbabane,Africa,1201670,1180655,1133936,1099920,1030496,854011,598564,442865,17364,69.2047,1.0079,0.02
12,ETH,Ethiopia,Addis Ababa,Africa,123379924,117190911,102471895,89237791,67031867,47878073,34945469,28308246,1104300,111.7268,1.0257,1.55
231,FLK,Falkland Islands,Stanley,South America,3780,3747,3408,3187,3080,2332,2240,2274,12173,0.3105,1.0043,0.0
209,FRO,Faroe Islands,Tórshavn,Europe,53090,52415,48816,48410,45660,47479,43054,38416,1393,38.112,1.0038,0.0
162,FJI,Fiji,Suva,Oceania,929766,920422,917200,905169,832509,780430,644582,527634,18272,50.8847,1.0056,0.01
118,FIN,Finland,Helsinki,Europe,5540745,5529468,5479461,5363271,5176209,4986545,4779418,4606621,338424,16.3722,1.0009,0.07
23,FRA,France,Paris,Europe,64626628,64480053,63809769,62444567,58665453,56412897,53713830,50523586,551695,117.1419,1.0015,0.81
184,GUF,French Guiana,Cayenne,South America,304557,290969,257026,228453,164351,113931,66825,46484,83534,3.6459,1.0239,0.0
183,PYF,French Polynesia,Papeete,Oceania,306279,301920,291787,283788,250927,211089,163591,117891,4167,73.5011,1.0074,0.0
146,GAB,Gabon,Libreville,Africa,2388992,2292573,2028517,1711105,1272935,983028,749078,597192,267668,8.9252,1.0204,0.03
142,GMB,Gambia,Banjul,Africa,2705992,2573995,2253133,1937275,1437539,1040616,718586,528731,10689,253.1567,1.025,0.03
131,GEO,Georgia,Tbilisi,Asia,3744385,3765912,3771132,3836831,4265172,5391636,5145843,4800426,69700,53.7214,0.9964,0.05
19,DEU,Germany,Berlin,Europe,83369843,83328988,82073226,81325090,81551677,79370196,77786703,78294583,357114,233.4544,0.9995,1.05
47,GHA,Ghana,Accra,Africa,33475870,32180401,28870939,25574719,19665502,15446982,11865246,8861895,238533,140.3406,1.0196,0.42
219,GIB,Gibraltar,Gibraltar,Europe,32649,32709,32520,31262,27741,27317,28734,26685,6,5441.5,0.9994,0.0
90,GRC,Greece,Athens,Europe,10384971,10512232,10806641,11033783,11038109,10302255,9307148,8544873,131990,78.68,0.9942,0.13
208,GRL,Greenland,Nuuk,North America,56466,56026,55895,56351,56184,55599,50106,45434,2166086,0.0261,1.004,0.0
193,GRD,Grenada,Saint George's,North America,125438,123663,118980,114039,107432,99047,94838,98794,344,364.6453,1.0066,0.0
178,GLP,Guadeloupe,Basse-Terre,North America,395752,395642,399089,403072,424067,391951,334234,318310,1628,243.0909,0.9992,0.0
191,GUM,Guam,Hagåtña,Oceania,171774,169231,167978,164905,160188,138263,110286,88300,549,312.8852,1.0073,0.0
68,GTM,Guatemala,Guatemala City,North America,17843908,17362718,16001107,14543121,11735894,9084780,6987767,5453208,108889,163.8725,1.0134,0.22
207,GGY,Guernsey,Saint Peter Port,Europe,63301,62794,61629,60782,59114,57727,52860,52656,78,811.5513,1.0037,0.0
75,GIN,Guinea,Conakry,Africa,13859341,13205153,11625998,10270728,8336967,6354145,4972609,4222374,245857,56.3716,1.0242,0.17
149,GNB,Guinea-Bissau,Bissau,Africa,2105566,2015828,1788919,1567220,1230849,973551,831462,591663,36125,58.2856,1.0218,0.03
164,GUY,Guyana,Georgetown,South America,808726,797202,755031,747932,759051,747116,778176,705261,214969,3.7621,1.0052,0.01
82,HTI,Haiti,Port-au-Prince,North America,11584996,11306801,10563757,9842880,8360225,6925331,5646676,4680812,27750,417.4773,1.012,0.15
89,HND,Honduras,Tegucigalpa,North America,10432860,10121763,9294505,8450933,6656725,5053234,3777990,2782753,112492,92.7431,1.015,0.13
104,HKG,Hong Kong,Hong Kong,Asia,7488865,7500958,7399838,7132438,6731195,5838574,4978544,3955072,1104,6783.3922,0.9992,0.09
94,HUN,Hungary,Budapest,Europe,9967308,9750573,9844246,9986825,10202055,10375989,10698679,10315366,93028,107.1431,1.0265,0.12
179,ISL,Iceland,Reykjavík,Europe,372899,366669,331060,318333,281462,255019,228263,204468,103000,3.6204,1.0069,0.0
2,IND,India,New Delhi,Asia,1417173173,1396387127,1322866505,1240613620,1059633675,870452165,696828385,557501301,3287590,431.0675,1.0068,17.77
4,IDN,Indonesia,Jakarta,Asia,275501339,271857970,259091970,244016173,214072421,182159874,148177096,115228394,1904569,144.6529,1.0064,3.45
17,IRN,Iran,Tehran,Asia,88550570,87290193,81790841,75373855,65544383,55793629,38520664,28449705,1648195,53.7258,1.0071,1.11
35,IRQ,Iraq,Baghdad,Asia,44496122,42556984,37757813,31264875,24628858,17658381,13653369,9811347,438317,101.5158,1.0221,0.56
125,IRL,Ireland,Dublin,Europe,5023109,4946119,4665760,4524585,3768950,3485374,3391387,2937637,70273,71.4799,1.0073,0.06
202,IMN,Isle of Man,Douglas,Europe,84519,84046,83593,83828,75562,68865,64022,55298,572,147.7605,1.003,0.0
98,ISR,Israel,Jerusalem,Asia,9038309,8757489,8007778,7328445,6116958,4803254,3744608,2907307,20770,435.1617,1.0155,0.11
25,ITA,Italy,Rome,Europe,59037474,59500579,60232906,59822450,56966397,56756561,56329482,53324036,301336,195.9191,0.9966,0.74
52,CIV,Ivory Coast,Yamoussoukro,Africa,28160542,26811790,23596741,21120042,16799670,11910540,8303809,5477086,322463,87.3295,1.0248,0.35
139,JAM,Jamaica,Kingston,North America,2827377,2820436,2794445,2733896,2612205,2392030,2135546,1859091,10991,257.2447,0.9999,0.04
11,JPN,Japan,Tokyo,Asia,123951692,125244761,127250933,128105431,126803861,123686321,117624196,105416839,377930,327.9753,0.9947,1.55
195,JEY,Jersey,Saint Helier,Europe,110778,108319,100561,96151,86192,82874,75124,68347,116,954.9828,1.0106,0.0
83,JOR,Jordan,Amman,Asia,11285869,10928721,9494246,6931258,5056174,3480587,2216903,1557374,89342,126.3221,1.0123,0.14
66,KAZ,Kazakhstan,Nursultan,Asia,19397998,18979243,17835909,16627837,15236253,16866563,14172710,12265305,2724900,7.1188,1.0105,0.24
27,KEN,Kenya,Nairobi,Africa,54027487,51985780,46851488,41517895,30851606,23162269,16187124,11473087,580367,93.0919,1.0193,0.68
192,KIR,Kiribati,Tarawa,Oceania,131232,126463,116707,107995,88826,75124,60813,57437,811,161.815,1.0183,0.0
129,KWT,Kuwait,Kuwait City,Asia,4268873,4360444,3908743,2943356,1934901,1674938,1493870,802786,17818,239.5821,1.0044,0.05
110,KGZ,Kyrgyzstan,Bishkek,Asia,6630623,6424874,5914980,5483774,4935182,4394734,3691209,3016384,199951,33.1612,1.0158,0.08
103,LAO,Laos,Vientiane,Asia,7529475,7319399,6787419,6323418,5430853,4314443,3297519,2675283,236800,31.7968,1.0141,0.09
151,LVA,Latvia,Riga,Europe,1850651,1897052,1991955,2101530,2392530,2689391,2572037,2397414,64559,28.666,0.9876,0.02
119,LBN,Lebanon,Beirut,Asia,5489739,5662923,6398940,4995800,4320642,3593700,2963702,2381791,10452,525.2334,0.9816,0.07
147,LSO,Lesotho,Maseru,Africa,2305825,2254100,2118521,2022747,1998630,1798997,1407672,1023481,30355,75.962,1.0107,0.03
121,LBR,Liberia,Monrovia,Africa,5302681,5087584,4612329,4019956,2895224,2209731,1932169,1463563,111369,47.6136,1.021,0.07
107,LBY,Libya,Tripoli,Africa,6812341,6653942,6192235,6491988,5154790,4236983,2962720,1909177,1759540,3.8717,1.0114,0.09
216,LIE,Liechtenstein,Vaduz,Europe,39327,38756,37355,35926,33026,28765,25003,21089,160,245.7937,1.0074,0.0
141,LTU,Lithuania,Vilnius,Europe,2750055,2820267,2963765,3139019,3599637,3785847,3521206,3210147,65300,42.1142,0.9869,0.03
168,LUX,Luxembourg,Luxembourg,Europe,647599,630399,569408,507070,435628,381267,363741,339342,2586,250.425,1.0129,0.01
167,MAC,Macau,Concelho de Macau,Asia,695168,676283,615239,557297,431896,350227,245332,247284,30,23172.2667,1.0125,0.01
50,MDG,Madagascar,Antananarivo,Africa,29611714,28225177,24850912,21731053,16216431,11882762,8948162,6639751,587041,50.4423,1.0241,0.37
62,MWI,Malawi,Lilongwe,Africa,20405317,19377061,16938942,14718422,11229387,9539665,6267369,4625141,118484,172.22,1.0259,0.26
45,MYS,Malaysia,Kuala Lumpur,Asia,33938221,33199993,31068833,28717731,22945150,17517054,13215707,10306508,330803,102.5934,1.0109,0.43
174,MDV,Maldives,Malé,Asia,523787,514438,435582,361575,282507,224957,164887,123243,300,1745.9567,1.0045,0.01
59,MLI,Mali,Bamako,Africa,22593590,21224040,18112907,15529181,11239101,8945026,7372581,6153587,1240192,18.2178,1.0314,0.28
173,MLT,Malta,Valletta,Europe,533286,515357,456579,418755,399212,365392,333587,315414,316,1687.6139,1.0124,0.01
215,MHL,Marshall Islands,Majuro,Oceania,41569,43413,49410,53416,54224,46047,31988,23969,181,229.663,0.9886,0.0
180,MTQ,Martinique,Fort-de-France,North America,367507,370391,383515,392181,432543,374271,333786,326428,1128,325.8041,0.9965,0.0
126,MRT,Mauritania,Nouakchott,Africa,4736139,4498604,3946220,3419461,2695003,2006027,1506694,1122198,1030700,4.5951,1.0263,0.06
157,MUS,Mauritius,Port Louis,Africa,1299469,1297828,1293153,1283330,1215930,1090290,954865,830115,2040,636.9946,1.0004,0.02
182,MYT,Mayotte,Mamoudzou,Africa,326101,305587,249545,211786,159215,92659,52233,35383,374,871.9278,1.0319,0.0
10,MEX,Mexico,Mexico City,North America,127504125,125998302,120149897,112532401,97873442,81720428,67705186,50289306,1964375,64.9082,1.0063,1.6
194,FSM,Micronesia,Palikir,Oceania,114164,112106,109462,107588,111709,98603,76299,58989,702,162.6268,1.0091,0.0
135,MDA,Moldova,Chisinau,Europe,3272996,3084847,3277388,3678186,4251573,4480199,4103240,3711140,33846,96.7026,1.0691,0.04
217,MCO,Monaco,Monaco,Europe,36469,36922,36760,33178,32465,30329,27076,24270,2,18234.5,0.9941,0.0
134,MNG,Mongolia,Ulaanbaatar,Asia,3398366,3294335,2964749,2702520,2450979,2161433,1697780,1293880,1564110,2.1727,1.0151,0.04
169,MNE,Montenegro,Podgorica,Europe,627082,629048,633966,631044,633324,621442,589324,530268,13812,45.4012,0.9988,0.01
230,MSR,Montserrat,Brades,North America,4390,4500,5059,4938,5138,10805,11452,11402,102,43.0392,0.9939,0.0
40,MAR,Morocco,Rabat,Africa,37457971,36688772,34680458,32464865,28554415,24570814,19678444,15274351,446550,83.883,1.0103,0.47
48,MOZ,Mozambique,Maputo,Africa,32969517,31178239,26843246,23073723,17768505,13303459,11413587,8411676,801590,41.1302,1.0278,0.41
26,MMR,Myanmar,Nay Pyi Taw,Asia,54179306,53423198,51483949,49390988,45538332,40099553,33465781,27284112,676578,80.0784,1.0071,0.68
145,NAM,Namibia,Windhoek,Africa,2567012,2489098,2282704,2099271,1819141,1369011,975994,754467,825615,3.1092,1.0146,0.03
225,NRU,Nauru,Yaren,Oceania,12668,12315,11185,10241,10377,9598,7635,6663,21,603.2381,1.0125,0.0
49,NPL,Nepal,Kathmandu,Asia,30547580,29348627,27610325,27161567,24559500,19616530,15600442,12501285,147181,207.5511,1.0171,0.38
71,NLD,Netherlands,Amsterdam,Europe,17564014,17434557,17041107,16617116,15899135,14944548,14130387,13037686,41850,419.6897,1.0036,0.22
185,NCL,New Caledonia,Nouméa,Oceania,289950,286403,283032,261426,221537,177264,148599,110982,18575,15.6097,1.0075,0.0
123,NZL,New Zealand,Wellington,Oceania,5185288,5061133,4590590,4346338,3855266,3397389,3147168,2824061,270467,19.1716,1.0108,0.07
106,NIC,Nicaragua,Managua,North America,6948392,6755895,6298598,5855734,5123222,4227820,3303309,2444767,130373,53.2962,1.0143,0.09
54,NER,Niger,Niamey,Africa,26207977,24333639,20128124,16647543,11622665,8370647,6173177,4669708,1267000,20.6851,1.0378,0.33
6,NGA,Nigeria,Abuja,Africa,218541212,208327405,183995785,160952853,122851984,95214257,72951439,55569264,923768,236.5759,1.0241,2.74
232,NIU,Niue,Alofi,Oceania,1934,1942,1847,1812,2074,2533,3637,5185,260,7.4385,0.9985,0.0
56,PRK,North Korea,Pyongyang,Asia,26069416,25867467,25258015,24686435,23367059,20799523,17973650,14996879,120538,216.2755,1.0038,0.33
150,MKD,North Macedonia,Skopje,Europe,2093599,2111072,2107962,2093828,2037936,2044174,1907023,1656783,25713,81.4218,0.9954,0.03
210,NFK,Northern Mariana Islands,Saipan,Oceania,49551,49587,51514,54087,80338,48002,17613,10143,464,106.7909,1.0014,0.0
120,NOR,Norway,Oslo,Europe,5434319,5379839,5190356,4889741,4491202,4241636,4085776,3875546,323802,16.7828,1.0058,0.07
127,OMN,Oman,Muscat,Asia,4576298,4543399,4191776,2881914,2344253,1804524,1017462,670693,309500,14.7861,1.0123,0.06
5,PAK,Pakistan,Islamabad,Asia,235824862,227196741,210969298,194454498,154369924,115414069,80624057,59290872,881912,267.4018,1.0191,2.96
222,PLW,Palau,Ngerulmud,Oceania,18055,17972,17794,18540,19726,15293,12252,11366,459,39.3355,1.0017,0.0
122,PSE,Palestine,Ramallah,Asia,5250072,5019401,4484614,3992278,3139954,2124609,1453620,1118241,6220,844.063,1.0227,0.07
128,PAN,Panama,Panama City,North America,4408581,4294396,3957099,3623617,3001731,2449968,1956987,1516188,75417,58.4561,1.0132,0.06
93,PNG,Papua New Guinea,Port Moresby,Oceania,10142619,9749640,8682174,7583269,5508297,3864972,3104788,2489059,462840,21.9139,1.0194,0.13
109,PRY,Paraguay,Asunción,South America,6780744,6618695,6177950,5768613,5123819,4059195,3078912,2408787,406752,16.6705,1.0115,0.09
44,PER,Peru,Lima,South America,34049588,33304756,30711863,29229572,26654439,22109099,17492406,13562371,1285216,26.4933,1.0099,0.43
13,PHL,Philippines,Manila,Asia,115559009,112190977,103031365,94636700,77958223,61558898,48419546,37435586,342353,337.5434,1.0147,1.45
37,POL,Poland,Warsaw,Europe,39857145,38428366,38553146,38597353,38504431,38064255,35521429,32482943,312679,127.4698,1.0404,0.5
92,PRT,Portugal,Lisbon,Europe,10270865,10298192,10365435,10588401,10300626,10007346,9785252,8683631,92090,111.5307,0.9981,0.13
136,PRI,Puerto Rico,San Juan,North America,3252407,3271564,3497335,3717922,3827108,3543776,3214568,2737619,8870,366.675,0.9989,0.04
143,QAT,Qatar,Doha,Asia,2695122,2760385,2414573,1713504,645937,441675,277450,118007,11586,232.6189,1.0026,0.03
114,COG,Republic of the Congo,Brazzaville,Africa,5970424,5702174,5064386,4437884,3134030,2385435,1829256,1396989,342000,17.4574,1.0231,0.07
161,REU,Reunion,Saint-Denis,Africa,974052,957822,922495,890130,785424,658992,551674,473925,2511,387.914,1.0082,0.01
64,ROU,Romania,Bucharest,Europe,19659267,19442038,19906079,20335211,21919876,22836234,22125224,19922618,238391,82.4665,1.0171,0.25
9,RUS,Russia,Moscow,Europe,144713314,145617329,144668389,143242599,146844839,148005704,138257420,130093010,17098242,8.4636,0.9973,1.81
76,RWA,Rwanda,Kigali,Africa,13776698,13146362,11642959,10309031,8109989,7319962,5247532,3896367,26338,523.0731,1.0234,0.17
228,BLM,Saint Barthelemy,Gustavia,North America,10967,10681,9643,8988,7082,5168,2983,2417,21,522.2381,1.0098,0.0
211,KNA,Saint Kitts and Nevis,Basseterre,North America,47657,47642,47790,47403,45461,40636,43097,44968,261,182.5939,1.0011,0.0
190,LCA,Saint Lucia,Castries,North America,179857,179237,175623,170935,159500,142301,121633,103090,616,291.9756,1.0011,0.0
220,MAF,Saint Martin,Marigot,North America,31791,32552,35020,36458,29610,28127,7776,5802,53,599.8302,0.9951,0.0
229,SPM,Saint Pierre and Miquelon,Saint-Pierre,North America,5862,5906,5978,6052,6274,6324,6106,5537,242,24.2231,0.9964,0.0
199,VCT,Saint Vincent and the Grenadines,Kingstown,North America,103948,104632,106482,109308,113813,112487,107480,98459,389,267.2185,0.9963,0.0
188,WSM,Samoa,Apia,Oceania,222382,214929,203571,194672,184008,168186,164905,142771,2842,78.2484,1.0165,0.0
218,SMR,San Marino,San Marino,Europe,33660,34007,33570,31608,26823,23132,21346,18169,61,551.8033,0.9975,0.0
187,STP,Sao Tome and Principe,São Tomé,Africa,227380,218641,201124,182138,143714,120343,97210,77583,964,235.8714,1.0192,0.0
41,SAU,Saudi Arabia,Riyadh,Asia,36408820,35997107,32749848,29411929,21547390,16004763,10171710,6106191,2149690,16.9368,1.0128,0.46
72,SEN,Senegal,Dakar,Africa,17316449,16436119,14356181,12530121,9704287,7536001,5703869,4367744,196722,88.025,1.0261,0.22
105,SRB,Serbia,Belgrade,Europe,7221365,7358005,7519496,7653748,7935022,7987529,7777010,7193533,88361,81.7257,0.9897,0.09
196,SYC,Seychelles,Victoria,Africa,107118,105530,99240,92409,80060,71057,65290,54379,452,236.9867,1.0061,0.0
102,SLE,Sierra Leone,Freetown,Africa,8605718,8233969,7314773,6436698,4584067,4325388,3367477,2778557,71740,119.957,1.022,0.11
113,SGP,Singapore,Singapore,Asia,5975689,5909869,5650018,5163590,4053602,3022209,2400729,2061831,710,8416.4634,1.0058,0.07
214,SXM,Sint Maarten,Philipsburg,North America,44175,43621,40205,33034,30489,27845,12243,6260,34,1299.2647,1.003,0.0
116,SVK,Slovakia,Bratislava,Europe,5643453,5456681,5424444,5396424,5376690,5261305,4973883,4522867,49037,115.0856,1.0359,0.07
148,SVN,Slovenia,Ljubljana,Europe,2119844,2117641,2080862,2057286,1984339,1986024,1901570,1741286,20273,104.5649,1.0002,0.03
166,SLB,Solomon Islands,Honiara,Oceania,724273,691191,612660,540394,429978,324171,233668,172833,28896,25.0648,1.0232,0.01
70,SOM,Somalia,Mogadishu,Africa,17597511,16537016,13763906,12026649,8721465,6999096,5892224,3720977,637657,27.5971,1.0312,0.22
24,ZAF,South Africa,Pretoria,Africa,59893885,58801927,55876504,51784921,46813266,39877570,29463549,22368306,1221037,49.0517,1.0084,0.75
29,KOR,South Korea,Seoul,Asia,51815810,51844690,50994401,48813042,46788591,44120039,38170501,32601143,100210,517.0722,0.9997,0.65
86,SSD,South Sudan,Juba,Africa,10913164,10606227,11194299,9714419,6114440,4750817,4192011,3342410,619745,17.6091,1.0153,0.14
30,ESP,Spain,Madrid,Europe,47558630,47363807,46431342,46572772,40741651,38889889,37491666,33792617,505992,93.9909,1.0015,0.6
61,LKA,Sri Lanka,Colombo,Asia,21832143,21715079,21336697,20668557,18776371,17204094,14943645,12388769,65610,332.7563,1.0027,0.27
32,SDN,Sudan,Khartoum,Africa,46874204,44440486,38171178,33739933,26298773,21090886,16673586,11305206,1886068,24.8529,1.0267,0.59
170,SUR,Suriname,Paramaribo,South America,618040,607065,575475,546080,478998,412756,375112,379918,163820,3.7727,1.0082,0.01
87,SWE,Sweden,Stockholm,Europe,10549347,10368969,9849349,9381729,8871043,8548406,8311763,8027702,450295,23.4276,1.0079,0.13
101,CHE,Switzerland,Bern,Europe,8740472,8638613,8281732,7822435,7182059,6711693,6319113,6181227,41284,211.7157,1.0056,0.11
60,SYR,Syria,Damascus,Asia,22125249,20772595,19205178,22337563,16307654,12408996,8898954,6319199,185180,119.4797,1.0376,0.28
57,TWN,Taiwan,Taipei,Asia,23893394,23821464,23512136,23083083,22194731,20586174,18100281,14957870,36193,660.1662,1.0014,0.3
95,TJK,Tajikistan,Dushanbe,Asia,9952787,9543207,8524063,7621779,6272998,5417860,4045965,2993019,143100,69.5513,1.0208,0.12
22,TZA,Tanzania,Dodoma,Africa,65497748,61704518,52542823,45110527,34463704,26206012,19297659,13618192,945087,69.3034,1.03,0.82
20,THA,Thailand,Bangkok,Asia,71697030,71475664,70294397,68270489,63066603,55228410,45737753,35791728,513120,139.7276,1.0013,0.9
155,TLS,Timor-Leste,Dili,Asia,1341296,1299995,1205813,1088486,878360,758106,642224,554021,14874,90.1772,1.0154,0.02
100,TGO,Togo,Lomé,Africa,8848699,8442580,7473229,6571855,5008035,3875947,2838110,2197383,56785,155.8281,1.0236,0.11
233,TKL,Tokelau,Nukunonu,Oceania,1871,1827,1454,1367,1666,1669,1647,1714,12,155.9167,1.0119,0.0
197,TON,Tonga,Nuku‘alofa,Oceania,106858,105254,106122,107383,102603,98727,96708,86484,747,143.0495,1.0079,0.0
153,TTO,Trinidad and Tobago,Port-of-Spain,North America,1531044,1518147,1460177,1410296,1332203,1266518,1127852,988890,5130,298.4491,1.0035,0.02
79,TUN,Tunisia,Tunis,Africa,12356117,12161723,11557779,10895063,9893316,8440023,6578156,5047404,163610,75.5218,1.0076,0.15
18,TUR,Turkey,Ankara,Asia,85341241,84135428,79646178,73195345,64113547,54324142,44089069,35540990,783562,108.9145,1.0067,1.07
111,TKM,Turkmenistan,Ashgabat,Asia,6430770,6250438,5766431,5267970,4569132,3720278,2862903,2201432,488100,13.1751,1.014,0.08
212,TCA,Turks and Caicos Islands,Cockburn Town,North America,45703,44276,36538,29726,18744,11709,7598,5665,948,48.2099,1.0131,0.0
227,TUV,Tuvalu,Funafuti,Oceania,11312,11069,10877,10550,9638,9182,7731,5814,26,435.0769,1.0096,0.0
31,UGA,Uganda,Kampala,Africa,47249585,44404611,37477356,32341728,24020697,17586630,13284026,10317212,241550,195.61,1.0304,0.59
38,UKR,Ukraine,Kiev,Europe,39701739,43909666,44982564,45683020,48879755,51589817,49973920,47279086,603500,65.7858,0.912,0.5
97,ARE,United Arab Emirates,Abu Dhabi,Asia,9441129,9287289,8916899,8481771,3275333,1900151,1014048,298084,83600,112.9322,1.0081,0.12
21,GBR,United Kingdom,London,Europe,67508936,67059474,65224364,62760039,58850043,57210442,56326328,55650166,242900,277.9289,1.0034,0.85
3,USA,United States,"Washington, D.C.",North America,338289857,335942003,324607776,311182845,282398554,248083732,223140018,200328340,9372610,36.0935,1.0038,4.24
200,VIR,United States Virgin Islands,Charlotte Amalie,North America,99465,100442,102803,106142,108185,100685,96640,63446,347,286.6427,0.9937,0.0
133,URY,Uruguay,Montevideo,South America,3422794,3429086,3402818,3352651,3292224,3117012,2953750,2790265,181034,18.9069,0.999,0.04
43,UZB,Uzbekistan,Tashkent,Asia,34627652,33526656,30949417,28614227,24925554,20579100,15947129,12011361,447400,77.3975,1.016,0.43
181,VUT,Vanuatu,Port-Vila,Oceania,326740,311685,276438,245453,192074,150882,118156,87019,12189,26.8061,1.0238,0.0
234,VAT,Vatican City,Vatican City,Europe,510,520,564,596,651,700,733,752,1,510.0,0.998,0.0
51,VEN,Venezuela,Caracas,South America,28301696,28490453,30529716,28715022,24427729,19750579,15210443,11355475,916445,30.882,1.0036,0.35
16,VNM,Vietnam,Hanoi,Asia,98186856,96648685,92191398,87411012,79001142,66912613,52968270,41928849,331212,296.4472,1.0074,1.23
226,WLF,Wallis and Futuna,Mata-Utu,Oceania,11572,11655,12182,13142,14723,13454,11315,9377,142,81.493,0.9953,0.0
172,ESH,Western Sahara,El Aaiún,Africa,575986,556048,491824,413296,270375,178529,116775,76371,266000,2.1654,1.0184,0.01
46,YEM,Yemen,Sanaa,Asia,33696614,32284046,28516545,24743946,18628700,13375121,9204938,6843607,527968,63.8232,1.0217,0.42
63,ZMB,Zambia,Lusaka,Africa,20017675,18927715,16248230,13792086,9891136,7686401,5720438,4281671,752612,26.5976,1.028,0.25
74,ZWE,Zimbabwe,Harare,Africa,16320537,15669666,14154937,12839771,11834676,10113893,7049926,5202918,390757,41.7665,1.0204,0.2"""
df = pd.read_csv(StringIO(csv_text)).rename(columns={
    'Country/Territory': 'Country', 'Area (km²)': 'Area',
    'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'
})
years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]
df_long = df.melt(id_vars=['CCA3','Country','Continent'],
                  value_vars=[f'{y} Population' for y in years],
                  var_name='Year', value_name='Population')
df_long['Year'] = df_long.Year.str[:4].astype(int)
assert df.Country.nunique() == len(df) == 234 and len(df_long) == 1872
blue, grey = '#0072B2', '#A9B2BA'
px.defaults.template = 'plotly_white'
continents = sorted(df.Continent.unique())
palette = dict(zip(continents, [blue, '#D58D00', '#695694', '#435466', '#54A8D0', '#B27598']))

def style(fig, title, height=460):
    fig.update_layout(title=dict(text=title, x=0.02), height=height,
                      margin=dict(l=25, r=30, t=78, b=42),
                      font=dict(size=13, color='#253541'), legend_title_text='')
    fig.update_xaxes(showgrid=False, zeroline=False)
    fig.update_yaxes(showgrid=False, zeroline=False)
    return fig

print(f'{len(df)} countries and territories · {len(years)} observation years')

234 countries and territories · 8 observation years


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


An indexed line chart starts each continent at 100 in 1970, so different starting populations are comparable. Africa reached 390 by 2022, or 3.90 times its starting population, the largest relative increase.

In [2]:
totals = df_long.groupby(['Continent','Year'], as_index=False).Population.sum().sort_values('Year')
totals['Index'] = 100 * totals.Population / totals.groupby('Continent').Population.transform('first')
fig = px.line(totals, x='Year', y='Index', color='Continent', markers=True,
              color_discrete_map={c: blue if c=='Africa' else grey for c in continents},
              hover_data={'Index': ':.1f'}, labels={'Index':'Population index (1970 = 100)'})
ends = totals.query('Year == 2022').sort_values('Index')
label_y = 0
for continent, value in ends[['Continent','Index']].itertuples(index=False, name=None):
    label_y = max(value, label_y + 17)
    fig.add_annotation(x=2022, y=value, text=f'{continent}  {value:.0f}',
                       showarrow=True, ax=2030, ay=label_y, axref='x', ayref='y',
                       arrowhead=0, arrowwidth=1, xanchor='left',
                       arrowcolor=grey, font_color=blue if continent=='Africa' else '#48545E')
fig.update_layout(showlegend=False)
fig.update_xaxes(range=[1969,2055], tickvals=[1970,1980,1990,2000,2010,2022], title=None)
style(fig, 'Africa grew to 3.90 times its 1970 population').show()

Africa grew to 3.90 times its 1970 population

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

Sorted horizontal bars show both rank and the size of each gain. Blue bars and star labels identify the four African countries in the top 15: Nigeria, Ethiopia, DR Congo and Egypt.

In [3]:
top = df.assign(Gain=df['2022 Population']-df['1970 Population']).nlargest(15,'Gain').copy()
top['Millions'] = top.Gain / 1e6
top['Label'] = top.Country + np.where(top.Continent.eq('Africa'),' ★','')
fig = px.bar(top, x='Millions', y='Label', orientation='h', text=top.Millions.map('{:.1f}m'.format),
             hover_name='Country', hover_data={'Gain':':,','Label':False,'Millions':False})
fig.update_traces(marker_color=[blue if c=='Africa' else grey for c in top.Continent],
                  textposition='outside', cliponaxis=False)
fig.update_yaxes(categoryorder='array', categoryarray=top.Label[::-1].tolist(), title=None)
fig.update_xaxes(title='People added, 1970–2022 (millions)', range=[0,990])
style(fig, 'India added 860 million people; four African countries rank in the top 15', 575).show()

India added 860 million people; four African countries rank in the top 15

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

A scatter plot on logarithmic axes keeps small territories and large countries visible together. Population generally increases with area in all six continents, but Greenland is sparsely populated for its size and Singapore is densely populated for its size.

In [4]:
# Pearson r compares the relationship within each continent on the plotted log scales.
corr = {c: np.log10(g.Area).corr(np.log10(g['2022 Population'])) for c,g in df.groupby('Continent')}
scatter = df.assign(Group=df.Continent.map({c:f'{c} (r={corr[c]:.2f})' for c in continents}),
                    Label=df.Country.where(df.Country.isin(['Greenland','Singapore','India']), ''))
fig = px.scatter(scatter, x='Area', y='2022 Population', color='Group', symbol='Group',
                 log_x=True, log_y=True, text='Label', hover_name='Country',
                 color_discrete_map={f'{c} (r={corr[c]:.2f})':palette[c] for c in continents},
                 labels={'Area':'Land area (km², log scale)','2022 Population':'Population (people, log scale)'})
fig.update_traces(marker_size=8, marker_opacity=.78, textposition='top center')
fig.update_layout(legend=dict(orientation='h', y=-.17))
style(fig, 'Larger countries generally have more people across all six continents', 560).show()

Larger countries generally have more people across all six continents

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

A box plot shows the median and middle half of the country or territory densities in each continent. Each entry counts once. North America has the highest median (221 people/km²), while South America has the lowest (18).

In [5]:
med = df.groupby('Continent').Density.median().sort_values(ascending=False)
fig = px.box(df, x='Continent', y='Density', points='outliers', hover_name='Country',
             category_orders={'Continent':med.index.tolist()}, log_y=True,
             labels={'Density':'People per km² (log scale)'})
fig.update_traces(marker_color=blue, fillcolor='rgba(0,114,178,.17)', quartilemethod='linear')
fig.update_xaxes(title=None)
style(fig, 'Median density ranges from 18 in South America to 221 in North America').show()

Median density ranges from 18 in South America to 221 in North America

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

A choropleth shows where population changed. Orange marks losses, blue marks gains, and the bins preserve differences among shrinking countries even though the largest gains exceed 300%. From 2000 to 2022, 34 countries or territories shrank, including Lithuania, Latvia and Moldova.

In [6]:
change = df.assign(Change=100*(df['2022 Population']/df['2000 Population']-1))
bands = ['Loss >10%', 'Loss ≤10%', 'No change', 'Gain ≤50%', 'Gain 50–100%', 'Gain >100%']
v = change.Change
change['Band'] = np.select([v < -10, v < 0, v == 0, v <= 50, v <= 100], bands[:5], default=bands[5])
# Correct the source's code for Northern Mariana Islands before mapping.
change['Map_code'] = change.CCA3.mask(change.Country.eq('Northern Mariana Islands'), 'MNP')
fig = px.choropleth(change, locations='Map_code', locationmode='ISO-3', color='Band',
                    category_orders={'Band':bands}, projection='natural earth',
                    color_discrete_map=dict(zip(bands,['#A75100','#F4B679','#E4E4E4','#A8D8EC','#4795BE','#145582'])),
                    hover_name='Country', hover_data={'Map_code':False,'Change':':+.1f','Band':False},
                    labels={'Change':'Population change (%)'})
fig.update_traces(marker_line_color='white', marker_line_width=.35)
fig.update_geos(showframe=False, showcoastlines=False, showland=True, landcolor='#E6E6E6')
fig.update_layout(legend=dict(orientation='h',y=-.02))
style(fig, 'Population fell in 34 places, including several in Eastern Europe', 485).show()

Population fell in 34 places, including several in Eastern Europe

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

A treemap nests countries within continents and sizes every tile by its 2022 population. Asia held 59.2% of the world's population. China and India together accounted for about 35.7% of the global total.

In [7]:
fig = px.treemap(df, path=[px.Constant('World'),'Continent','Country'], values='2022 Population',
                 color='Continent', color_discrete_map=palette)
fig.update_traces(root_color='#E7EBEE', textinfo='label+percent root',
                  hovertemplate='%{label}<br>%{value:,.0f} people<br>%{percentRoot:.2%} of world<extra></extra>')
style(fig, 'Asia held 4.72 billion of the world’s 7.97 billion people in 2022', 590).show()

Asia held 4.72 billion of the world’s 7.97 billion people in 2022

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

A 100% stacked bar for each observed year shows how the population divides among the four groups. Africa rose from 9.9% to 17.9%, Europe fell from 17.8% to 9.3%, and Asia edged up from 58.1% to 59.2%.

In [8]:
groups = ['Africa','Asia','Europe','Rest of the world']
shares = totals.assign(Group=totals.Continent.where(totals.Continent.isin(groups[:3]), groups[3]))
shares = shares.groupby(['Year','Group'], as_index=False).Population.sum()
shares['Share'] = 100*shares.Population/shares.groupby('Year').Population.transform('sum')
assert np.allclose(shares.groupby('Year').Share.sum(), 100)
shares['Period'] = shares.Year.astype(str)
fig = px.bar(shares, x='Period', y='Share', color='Group', barmode='stack',
             category_orders={'Group':groups}, color_discrete_map=dict(zip(groups,[blue,'#BBDAE9','#D99945','#D4D7DA'])),
             text=shares.Share.map('{:.1f}%'.format), hover_data={'Share':':.1f','Population':':,'})
fig.update_traces(textposition='inside', insidetextanchor='middle')
fig.update_xaxes(title=None, type='category')
fig.update_yaxes(range=[0,100], ticksuffix='%', title=None)
fig.update_layout(legend=dict(orientation='h',y=-.12))
style(fig, 'Africa gained eight percentage points of world population as Europe’s share fell', 485).show()

Africa gained eight percentage points of world population as Europe’s share fell